# 02 · Filtros à mão

Aula **"Filtros à mão"**: o filtro é um **kernel**, uma pequena matriz de pesos que passa por
cada pixel e troca o valor dele pela soma ponderada da vizinhança. Neste notebook:

1. **kernel e vizinhança**: um pixel calculado na mão e a imagem inteira em NumPy, conferida com o OpenCV;
2. **galeria de kernels** num raio-X: média, gaussiano, realce, Sobel;
3. **a trinca no ruído**: qual suavização limpa qual ruído, medido em PSNR em 200 raio-X;
4. **Sobel**: onde está a borda do coração, num perfil horizontal;
5. **Canny**: os dois limiares e a escolha automática;
6. **CLAHE**: equalização global contra adaptativa, e o que ela faz com a variação entre exames;
7. **o limite**: o que filtro nenhum resolve, e por que os próximos notebooks existem.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import auditoria, filtros

configurar_estilo()
SECAO = "02-filtros"

baixar_base()
catalogo = dividir(listar_imagens())
treino = catalogo[catalogo["conjunto"] == "treino"].reset_index(drop=True)

# Raio-X de referência: cardiomegalia, sem fios, coração bem visível.
EXEMPLO = "train/true/1298"
img = carregar_imagens(catalogo.loc[catalogo["id"] == EXEMPLO, "caminho"])[0]

def mostrar(ax, imagem, titulo, **kw):
    ax.imshow(imagem, cmap=kw.pop("cmap", "gray"), **({"vmin": 0, "vmax": 255} | kw))
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")

## 1. Kernel e vizinhança

Um pixel e seus 8 vizinhos, o kernel gaussiano 3×3 e a conta que dá o novo valor do pixel.

In [ ]:
lin, col = 64, 64
vizinhanca = img[lin - 1 : lin + 2, col - 1 : col + 2].astype(np.float32)
kernel = filtros.GAUSS_3
produto = vizinhanca * kernel
print("Vizinhança:\n", vizinhanca.astype(int))
print("\nKernel gaussiano (×16):\n", (kernel * 16).astype(int))
print(f"\nNovo valor = soma(vizinhança × kernel) = {produto.sum():.2f}  (era {img[lin, col]})")

a_mao = filtros.correlacionar(img, kernel)
opencv = cv2.filter2D(img.astype(np.float32), -1, kernel, borderType=cv2.BORDER_REFLECT_101)
print(f"Pixel ({lin},{col}) pela função NumPy: {a_mao[lin, col]:.2f}")
print(f"Maior diferença NumPy × OpenCV na imagem inteira: {np.abs(a_mao - opencv).max():.2e}")

## 2. Galeria de kernels

O mesmo raio-X passando pelos kernels escritos à mão. Média e gaussiano **borram**; o realce
**aguça** (centro 5, vizinhos −1); Sobel x e y acendem onde a intensidade **muda** na horizontal e
na vertical (cinza = sem mudança, claro e escuro = subida e descida).

In [ ]:
fig, eixos = plt.subplots(1, 6, figsize=(16, 3.4))
mostrar(eixos[0], img, "Original")
for ax, (nome, k) in zip(eixos[1:], filtros.KERNELS.items()):
    saida = filtros.correlacionar(img, k)
    if nome.startswith("Sobel"):
        lim = np.abs(saida).max()
        mostrar(ax, saida, nome, vmin=-lim, vmax=lim)
    else:
        mostrar(ax, filtros.para_uint8(saida), nome)
salvar_figura(fig, SECAO, "galeria-kernels")
plt.show()

## 3. A trinca no ruído

Dois ruídos de verdade em raio-X: **gaussiano** (chiado do sensor, todo pixel um pouco) e
**sal e pimenta** (pixel morto, poeira, poucos pixels muito). Três suavizações. Medida: **PSNR**
contra o raio-X limpo, em dB (maior = mais perto do original), média de 200 raio-X de treino.

In [ ]:
rng = np.random.default_rng(config.SEMENTE)
amostra = carregar_imagens(treino.sample(200, random_state=config.SEMENTE)["caminho"])
ruidos = {
    "Gaussiano σ=15": lambda im: filtros.ruido_gaussiano(im, 15, rng),
    "Sal e pimenta 5 %": lambda im: filtros.ruido_sal_pimenta(im, 0.05, rng),
}

linhas = []
for nome_ruido, sujar in ruidos.items():
    for limpa in amostra:
        suja = sujar(limpa)
        linhas.append({"ruido": nome_ruido, "filtro": "sem filtro", "psnr": filtros.psnr(limpa, suja)})
        for nome_filtro, aplicar in filtros.SUAVIZADORES.items():
            linhas.append({"ruido": nome_ruido, "filtro": nome_filtro, "psnr": filtros.psnr(limpa, aplicar(suja))})
psnr = pd.DataFrame(linhas).pivot_table(index="filtro", columns="ruido", values="psnr", aggfunc="mean")
psnr = psnr.loc[["sem filtro", *filtros.SUAVIZADORES]]
salvar_tabela(psnr, SECAO, "psnr-ruido-filtro")
psnr.round(1)

In [ ]:
fig, eixos = plt.subplots(2, 4, figsize=(13, 6.6))
for linha, (nome_ruido, sujar) in enumerate(ruidos.items()):
    suja = sujar(img)
    mostrar(eixos[linha, 0], suja, f"{nome_ruido}\nPSNR {filtros.psnr(img, suja):.1f} dB")
    for ax, (nome_filtro, aplicar) in zip(eixos[linha, 1:], filtros.SUAVIZADORES.items()):
        limpa = aplicar(suja)
        mostrar(ax, limpa, f"{nome_filtro}\nPSNR {filtros.psnr(img, limpa):.1f} dB")
salvar_figura(fig, SECAO, "ruido-e-suavizacao", psnr)
plt.show()

## 4. Sobel: onde está a borda do coração

Uma linha horizontal atravessando o coração. A intensidade sobe ao sair do pulmão (escuro)
e entrar no coração (claro), e desce do outro lado. O Sobel x transforma essas **mudanças** em
picos: as bordas do coração são onde |gx| é grande dentro do tórax.

In [ ]:
LINHA = 66
gx, gy, magnitude = filtros.sobel(cv2.GaussianBlur(img, (5, 5), 0))
perfil = pd.DataFrame({"intensidade": img[LINHA].astype(int), "sobel_x": gx[LINHA]})

fig = plt.figure(figsize=(13, 6.5))
grade = fig.add_gridspec(2, 3, height_ratios=[1.3, 1])
mostrar(fig.add_subplot(grade[0, 0]), img, "Raio-X e a linha do perfil")
fig.axes[0].axhline(LINHA, color=config.CORAL, lw=1.5)
lim = np.abs(gx).max()
mostrar(fig.add_subplot(grade[0, 1]), gx, "Sobel x", cmap="RdBu_r", vmin=-lim, vmax=lim)
mostrar(fig.add_subplot(grade[0, 2]), magnitude, "Magnitude do gradiente", cmap="magma",
        vmin=0, vmax=np.percentile(magnitude, 99))
ax = fig.add_subplot(grade[1, :])
ax.plot(perfil.index, perfil["intensidade"], color=config.AZUL_PROFUNDO, label="intensidade")
ax2 = ax.twinx()
ax2.plot(perfil.index, perfil["sobel_x"], color=config.CORAL, label="Sobel x")
ax2.axhline(0, color="#999", lw=0.8)
ax2.grid(False)
ax.set(xlabel="coluna (pixel)", ylabel="intensidade", title=f"Perfil da linha {LINHA}")
ax2.set_ylabel("Sobel x")
fig.legend(loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.03))
salvar_figura(fig, SECAO, "sobel-perfil", perfil)
plt.show()

## 5. Canny: dois limiares

O Canny afina o Sobel numa linha de 1 pixel e decide o que é borda com **dois limiares**:
acima do alto é borda; entre os dois só é borda se encostar numa borda forte; abaixo do baixo
é descartado. Limiares baixos trazem costela, fio e ruído; altos apagam a borda do coração.
A versão automática tira os limiares da mediana de cada raio-X, que muda com a exposição.

In [ ]:
suave = cv2.GaussianBlur(img, (5, 5), 0)
pares = [(20, 60), (50, 120), (100, 200)]
auto, (baixo, alto) = filtros.canny_auto(img)

fig, eixos = plt.subplots(1, 5, figsize=(16, 3.6))
mostrar(eixos[0], img, "Original")
contagem = {}
for ax, (lo, hi) in zip(eixos[1:], pares):
    bordas = cv2.Canny(suave, lo, hi)
    contagem[f"{lo}/{hi}"] = int((bordas > 0).sum())
    mostrar(ax, bordas, f"Canny {lo}/{hi}")
contagem[f"auto {baixo}/{alto}"] = int((auto > 0).sum())
mostrar(eixos[4], auto, f"Automático {baixo}/{alto}")
salvar_figura(fig, SECAO, "canny-limiares",
              pd.DataFrame({"pixels_de_borda": contagem}))
plt.show()

## 6. CLAHE

**Equalização global** (`equalizeHist`) espalha o histograma da imagem inteira. **CLAHE** faz isso
por blocos 8×8, com limite de contraste para não amplificar ruído. É o passo do exemplo de
ultrassom da aula.

In [ ]:
global_eq = cv2.equalizeHist(img)
adaptativa = filtros.clahe(img)

fig, eixos = plt.subplots(2, 3, figsize=(12, 7), gridspec_kw={"height_ratios": [2.2, 1]})
hist = {}
for j, (nome, im) in enumerate([("Original", img), ("Equalização global", global_eq), ("CLAHE", adaptativa)]):
    mostrar(eixos[0, j], im, nome)
    h, _ = np.histogram(im, bins=np.arange(257))
    hist[nome] = h
    eixos[1, j].bar(np.arange(256), h, width=1, color=config.AZUL_COBALTO)
    eixos[1, j].set(xlim=(0, 255), yticks=[])
salvar_figura(fig, SECAO, "clahe", pd.DataFrame(hist))
plt.show()

Raio-X de aparelhos e pacientes diferentes chegam com brilho e contraste diferentes. Se o
pré-processamento os aproxima, a rede depende menos da exposição. Medida: o **desvio entre
imagens** do brilho e do contraste, antes e depois, nos 200 raio-X.

In [ ]:
versoes = {
    "original": amostra,
    "equalização global": np.stack([cv2.equalizeHist(im) for im in amostra]),
    "CLAHE": np.stack([filtros.clahe(im) for im in amostra]),
    "gaussiano + CLAHE (padrão)": np.stack([filtros.preprocessar(im) for im in amostra]),
}
variacao = pd.DataFrame({
    nome: auditoria.qualidade(ims)[["brilho", "contraste", "nitidez"]].std() for nome, ims in versoes.items()
}).T.add_prefix("desvio_entre_imagens_")
salvar_tabela(variacao, SECAO, "variacao-entre-imagens")
variacao.round(1)

## 7. O limite

* **Kernel na mão = OpenCV:** a correlação escrita em NumPy deu diferença zero contra o
  `cv2.filter2D` na imagem inteira. Filtro não é mágica: é soma ponderada da vizinhança.
* **Cada ruído tem seu filtro:** no ruído gaussiano as três suavizações empatam (≈ +1,5 dB,
  de 25,0 para 26,3–26,8 dB). No **sal e pimenta** a **mediana** vence com folga: 17,7 → 28,4 dB,
  contra 23,8 da média e 24,5 do gaussiano. A média espalha o pixel estragado pelos vizinhos;
  a mediana simplesmente o ignora.
* **Sobel acha mudança, não coração:** no perfil, a borda direita do coração é um pico agudo,
  mas a esquerda é uma **rampa suave**, fraca para o gradiente. As bordas mais fortes da imagem
  são o contorno dos pulmões, o diafragma e a borda do filme.
* **Canny depende dos limiares:** de 20/60 para 100/200 os pixels de borda caem de 2.642 para
  1.363. O automático pela mediana (99/196) cai perto do par alto. Em nenhum par a borda do
  coração sai fechada.
* **CLAHE contra equalização global:** a global é a que mais **iguala os exames entre si**
  (desvio do brilho entre imagens 24,9 → 6,5), mas lava o contraste dentro do tórax. O CLAHE
  realça o detalhe local e iguala menos (24,9 → 20,9). O padrão do projeto, gaussiano 3×3 +
  CLAHE (`filtros.preprocessar`), mantém o detalhe e ainda reduz a variação de nitidez entre
  exames (332 → 128).

**O que filtro nenhum resolve:** o filtro só olha uma vizinhança 3×3 ou 5×5. Ele não sabe
o que é coração, costela ou diafragma. Para **medir** o coração é preciso separar regiões
(limiar e morfologia, notebook 04); para **reconhecer** a cardiomegalia sem regra escrita,
deixar a rede aprender os próprios filtros (CNN, notebook 05).